# Chapter 13: Retrieval-Augmented Generation (RAG)
## Notebook 02 — The RAG Pipeline

Now that you have built RAG from scratch, this notebook walks through the **engineering decisions** that turn a toy demo into a practical pipeline:

| Topic | Section |
|-------|--------|
| Chunking strategies (fixed / sliding / sentence / semantic) | §1 |
| Embedding model choices (with TF-IDF fallback) | §2 |
| Vector store options (FAISS / Chroma / in-memory) | §3 |
| Full pipeline: load → chunk → embed → index → retrieve → generate | §4 |
| Reranking with a cross-encoder (concept + lexical proxy) | §5 |
| Prompt assembly and citation handling | §6 |

**Estimated time:** 2.5 hours

---
*Generated by Berta AI*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

# Load the chapter corpus
CORPUS_PATH = os.path.join('..', 'datasets', 'sample_corpus.txt')
with open(CORPUS_PATH) as f:
    raw = f.read()

# Parse "[doc-NNN] text" paragraphs into a {doc_id: text} dict
import re
pattern = re.compile(r'^\[(doc-\d+)\]\s*(.+)$', re.MULTILINE | re.DOTALL)
documents = {}
for para in re.split(r'\n\s*\n', raw):
    m = pattern.match(para.strip())
    if m:
        documents[m.group(1)] = m.group(2).strip()
print("Loaded", len(documents), "documents")
print("First doc:", list(documents.values())[0][:100])

---
## 1. Chunking Strategies

Long documents are too big to index as a single vector — the vector loses topical resolution and the LLM cannot attend to all of it. We **chunk** them into smaller passages first.

The chapter ships four strategies in `scripts/chunking.py`:

- **Fixed-size** — split by character count. Simple but breaks sentences.
- **Sliding window** — overlapping windows of tokens. Preserves context across boundaries.
- **Sentence-packed** — group whole sentences up to a token budget. Respects natural language.
- **Semantic** — start a new chunk when consecutive sentences are *dissimilar* (TF-IDF cosine).

Below we apply all four to a single document and compare.

In [ ]:
from chunking import (
    fixed_size_chunks, sliding_window_chunks,
    sentence_chunks, semantic_chunks, Chunker,
)

doc_id = "doc-001"
text = documents[doc_id]
print("Document length:", len(text), "chars,", len(text.split()), "tokens\n")

strategies = {
    "fixed":    fixed_size_chunks(text, chunk_size=120, doc_id=doc_id),
    "sliding":  sliding_window_chunks(text, window_tokens=20, overlap_tokens=5, doc_id=doc_id),
    "sentence": sentence_chunks(text, max_tokens=20, doc_id=doc_id),
    "semantic": semantic_chunks(text, similarity_threshold=0.2, max_tokens=40, doc_id=doc_id),
}

for name, chunks in strategies.items():
    print(f"{name:10} -> {len(chunks)} chunks")
    for c in chunks[:2]:
        print(f"   [{c.chunk_id}] {c.text[:80]}")
    print()

### How to choose a chunk size

There is no single right answer, but two rules of thumb:

- Embedding-model context windows are typically 512 tokens — chunks much larger than that lose detail.
- Smaller chunks improve retrieval precision but require more chunks at retrieval time. **256 tokens with ~32 token overlap** is a good starting point.

Let's measure how chunk count varies with chunk size for the full corpus.

In [ ]:
sizes = [60, 120, 240, 480]
counts = []
for s in sizes:
    ch = Chunker(strategy="sliding", window_tokens=s, overlap_tokens=s // 5)
    total = sum(len(ch.chunk(t, doc_id=d)) for d, t in documents.items())
    counts.append(total)

plt.bar([str(s) for s in sizes], counts, color="steelblue")
plt.xlabel("Window tokens")
plt.ylabel("Total chunks across corpus")
plt.title("Chunk count vs window size")
plt.tight_layout()
plt.show()

print(dict(zip(sizes, counts)))

---
## 2. Embedding Model Choices

The chapter supports two embedders out of the box:

1. **`sentence-transformers`** — high quality, ~384-dim vectors, runs on CPU. *Optional install.*
2. **`TfidfEmbedder`** — TF-IDF + truncated SVD. Pure scikit-learn. The CI default.

The pipeline auto-falls back to TF-IDF if `sentence-transformers` is not installed, so notebooks always run.

In [ ]:
# Try sentence-transformers; fall back gracefully.
try:
    from sentence_transformers import SentenceTransformer
    _ST_AVAILABLE = True
    print("sentence-transformers available")
except Exception as e:
    _ST_AVAILABLE = False
    print("sentence-transformers NOT installed -> using TF-IDF fallback")
    print("Install with: pip install sentence-transformers")

class STEmbedder:
    """Thin wrapper that matches the TfidfEmbedder API."""
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()

    def fit(self, texts):
        return self  # nothing to fit for ST

    def encode(self, texts):
        return np.asarray(self.model.encode(list(texts)), dtype=np.float32)

    def encode_query(self, text):
        return self.encode([text])[0]

from rag_pipeline import TfidfEmbedder
embedder = STEmbedder() if _ST_AVAILABLE else TfidfEmbedder(dim=128)
print("Using:", type(embedder).__name__, "dim=", getattr(embedder, "dim", "?"))

---
## 3. Vector Store Options

In production you'll usually pick one of these:

- **FAISS** — Meta's library; exact and approximate ANN at billion-scale.
- **Chroma** — Pythonic local-first DB with metadata filtering.
- **Pinecone / Weaviate / Qdrant** — managed services.

For this chapter we use the in-memory NumPy store from `scripts/vectorstore.py` because it works **anywhere** with no extra installs. The API is intentionally compatible with FAISS.

In [ ]:
# FAISS sketch (commented unless faiss is installed)
try:
    import faiss
    print("faiss is available, dim 4 demo:")
    index = faiss.IndexFlatIP(4)
    vecs = np.random.randn(5, 4).astype('float32')
    faiss.normalize_L2(vecs)
    index.add(vecs)
    q = np.random.randn(1, 4).astype('float32'); faiss.normalize_L2(q)
    D, I = index.search(q, 2)
    print("  faiss top-2:", I[0].tolist(), "scores:", D[0].tolist())
except Exception:
    print("faiss not installed -> using InMemoryVectorStore")
    print("Install: pip install faiss-cpu")

# Chroma sketch (just imports)
try:
    import chromadb
    print("chromadb is available")
except Exception:
    print("chromadb not installed (optional). Install: pip install chromadb")

In [ ]:
# Default in-memory store
from vectorstore import InMemoryVectorStore

# Fit the embedder on the corpus first
embedder.fit(list(documents.values()))
embs = embedder.encode(list(documents.values()))
store = InMemoryVectorStore(dim=embs.shape[1])
store.add(
    embeddings=embs,
    chunk_ids=list(documents.keys()),
    texts=list(documents.values()),
)
print("Indexed", len(store), "documents in", embs.shape[1], "dims")

---
## 4. End-to-End Pipeline

`RAGPipeline` wires together: chunker → embedder → vector store → (reranker) → LLM. Drop in any embedder or LLM that exposes the small duck-typed surface area.

In [ ]:
from rag_pipeline import RAGPipeline, MockLLM
from chunking import Chunker

pipe = RAGPipeline(
    chunker=Chunker(strategy="sentence", max_tokens=80),
    embedder=embedder,
    llm_client=MockLLM(),
    top_k=3,
)
n_chunks = pipe.index_documents(documents)
print("Indexed", n_chunks, "chunks from", len(documents), "documents")

response = pipe.answer("How does hybrid search combine dense and sparse retrieval?")
print("\n=== Answer ===")
print(response.answer)
print("\n=== Retrieved chunks ===")
for c in response.contexts:
    print(f"  [{c.chunk_id}] score={c.score:.3f}  {c.text[:80]}")

---
## 5. Reranking

A **bi-encoder** (the embedder above) scores query and document independently — fast, but not very precise. A **cross-encoder** takes both inputs together and outputs a single relevance score — slow, but highly accurate.

The standard recipe: **retrieve a wide candidate set with a bi-encoder, then rerank the top 20–50 with a cross-encoder.** Here, we use a deterministic *lexical* reranker as a stand-in so the notebook runs without any extra installs.

In [ ]:
def lexical_reranker(query, hits, top_k=None):
    """Boost candidates that contain query keywords (simple and offline)."""
    q_terms = set(t.lower() for t in query.split() if len(t) > 3)
    out = []
    for h in hits:
        terms = set(t.lower() for t in h.text.split())
        overlap = len(q_terms & terms)
        # Combine original cosine score with overlap weight
        from vectorstore import SearchResult
        new_score = h.score + 0.05 * overlap
        out.append(SearchResult(chunk_id=h.chunk_id, score=new_score,
                                text=h.text, metadata=h.metadata))
    out.sort(key=lambda r: -r.score)
    return out if top_k is None else out[:top_k]

# Plug it in
pipe.reranker = lexical_reranker
print("\nWith lexical reranker:")
r = pipe.answer("What does Reciprocal Rank Fusion do?")
for c in r.contexts:
    print(f"  [{c.chunk_id}] score={c.score:.3f}  {c.text[:80]}")

### Real cross-encoder reranking

```python
# Optional, requires sentence-transformers:
# from sentence_transformers import CrossEncoder
# ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
# def ce_reranker(query, hits, top_k=None):
#     pairs = [[query, h.text] for h in hits]
#     scores = ce.predict(pairs)
#     reordered = sorted(zip(hits, scores), key=lambda x: -x[1])
#     return [h for h, _ in reordered][: top_k or len(reordered)]
```

---
## 6. Prompt Assembly and Citations

A well-built RAG prompt does three jobs:

1. Tells the model to **use only the retrieved context**.
2. Tells the model to **cite sources** by their chunk identifier.
3. Tells the model to **refuse** when the context is missing the answer.

Here is the prompt template `RAGPipeline` builds. Notice the `[chunk_id]` prefixes — they make the LLM's citations checkable later.

In [ ]:
query = "What is HyDE in retrieval?"
contexts = pipe.retrieve(query, top_k=3)
prompt = pipe._build_prompt(query, contexts)
print(prompt)

In [ ]:
# A simple post-hoc citation extractor: pull bracketed ids out of the answer.
import re

resp = pipe.answer(query)
print("Answer:\n", resp.answer, "\n")

cited = re.findall(r'\[([^\]]+)\]', resp.answer)
print("Citations parsed from answer:", cited)
# In production, verify each citation actually appears in the retrieved contexts:
ctx_ids = {c.chunk_id for c in resp.contexts}
unsupported = [c for c in cited if c not in ctx_ids]
print("Unsupported citations (should be empty):", unsupported)

---
## 7. Key Takeaways

- **Chunking choice** is a hyperparameter. Sentence and semantic chunking usually beat fixed-size for natural text.
- **Embedding choice** is the biggest quality lever: sentence-transformers >> TF-IDF for most tasks. Always have a fallback for CI.
- **Vector store choice** is largely about scale and ops. Start in-memory; graduate to FAISS or a managed DB when you outgrow it.
- **Bi-encoder + cross-encoder rerank** is the standard high-quality retrieval stack.
- **Prompts must instruct citation and refusal**, otherwise the LLM happily fabricates.

Next: **Notebook 03** — hybrid search, query rewriting, faithfulness evaluation, and production concerns.

---
*Generated by Berta AI*